# Model Evaluation & Comparison

This notebook provides a comprehensive comparison of all trained models:
- **Teacher**: BERT-base-uncased (109.5M parameters)
- **Student Baseline**: Compact transformer trained with hard labels only
- **Student Distilled**: Compact transformer trained with knowledge distillation

We analyze performance, compression efficiency, and training dynamics.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 150

# Load results
with open("../outputs/teacher/results.json") as f:
    teacher_results = json.load(f)
with open("../outputs/student/baseline/results.json") as f:
    baseline_results = json.load(f)
with open("../outputs/student/distill/results.json") as f:
    distill_results = json.load(f)

# Load epoch-level metrics
teacher_metrics = pd.read_csv("../outputs/teacher/metrics.csv")
baseline_metrics = pd.read_csv("../outputs/student/baseline/metrics.csv")
distill_metrics = pd.read_csv("../outputs/student/distill/metrics.csv")

print("All results loaded successfully.")

## 1. Results Summary

In [ ]:
# Build comparison table
summary = pd.DataFrame({
    "Model": ["Teacher (BERT-base)", "Student Baseline", "Student Distilled"],
    "Parameters": [
        teacher_results["final_results"]["num_parameters"],
        baseline_results["final_results"]["num_parameters"],
        distill_results["final_results"]["num_parameters"],
    ],
    "Size (MB)": [
        teacher_results["final_results"]["model_size_mb"],
        baseline_results["final_results"]["model_size_mb"],
        distill_results["final_results"]["model_size_mb"],
    ],
    "Test Accuracy": [
        teacher_results["final_results"]["test"]["accuracy"],
        baseline_results["final_results"]["test"]["accuracy"],
        distill_results["final_results"]["test"]["accuracy"],
    ],
    "Test F1 (Macro)": [
        teacher_results["final_results"]["test"]["f1_macro"],
        baseline_results["final_results"]["test"]["f1_macro"],
        distill_results["final_results"]["test"]["f1_macro"],
    ],
    "Training Time (min)": [
        teacher_results["total_training_time_minutes"],
        baseline_results["total_training_time_minutes"],
        distill_results["total_training_time_minutes"],
    ],
})

summary["Compression"] = summary["Parameters"].iloc[0] / summary["Parameters"]
summary["Teacher Retention (%)"] = summary["Test Accuracy"] / summary["Test Accuracy"].iloc[0] * 100

print("=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)
display(summary.style.format({
    "Parameters": "{:,.0f}",
    "Size (MB)": "{:.1f}",
    "Test Accuracy": "{:.4f}",
    "Test F1 (Macro)": "{:.4f}",
    "Training Time (min)": "{:.1f}",
    "Compression": "{:.1f}x",
    "Teacher Retention (%)": "{:.2f}%",
}))

## 2. Performance Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models = ["Teacher\n(BERT-base)", "Student\n(Baseline)", "Student\n(Distilled)"]
colors = ["#4C72B0", "#DD8452", "#55A868"]

# Accuracy comparison
accs = [r["final_results"]["test"]["accuracy"] for r in [teacher_results, baseline_results, distill_results]]
bars = axes[0].bar(models, accs, color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_ylabel("Test Accuracy")
axes[0].set_title("Test Accuracy")
axes[0].set_ylim(0.97, 1.0)
for bar, val in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f"{val:.4f}", ha="center", va="bottom", fontsize=10)

# F1 comparison
f1s = [r["final_results"]["test"]["f1_macro"] for r in [teacher_results, baseline_results, distill_results]]
bars = axes[1].bar(models, f1s, color=colors, edgecolor="black", linewidth=0.5)
axes[1].set_ylabel("F1 Score (Macro)")
axes[1].set_title("Test F1 (Macro)")
axes[1].set_ylim(0.97, 1.0)
for bar, val in zip(bars, f1s):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f"{val:.4f}", ha="center", va="bottom", fontsize=10)

# Model size comparison
sizes = [r["final_results"]["model_size_mb"] for r in [teacher_results, baseline_results, distill_results]]
bars = axes[2].bar(models, sizes, color=colors, edgecolor="black", linewidth=0.5)
axes[2].set_ylabel("Model Size (MB)")
axes[2].set_title("Model Size")
for bar, val in zip(bars, sizes):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f"{val:.1f} MB", ha="center", va="bottom", fontsize=10)

plt.suptitle("Teacher vs. Student Models: Performance & Efficiency", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: reports/model_comparison.png")

## 3. Training Dynamics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Teacher training curves
axes[0, 0].plot(teacher_metrics["epoch"], teacher_metrics["train_loss"], "b-o", markersize=4, label="Train Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].set_title("Teacher: Training Loss")
axes[0, 0].legend()

axes[0, 1].plot(teacher_metrics["epoch"], teacher_metrics["val_acc"], "r-o", markersize=4, label="Val Accuracy")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].set_title("Teacher: Validation Accuracy")
axes[0, 1].set_ylim(0.98, 1.0)
axes[0, 1].legend()

# Student comparison: validation accuracy
axes[1, 0].plot(baseline_metrics["epoch"], baseline_metrics["val_acc"], "--o", color="#DD8452",
               markersize=3, label="Baseline")
axes[1, 0].plot(distill_metrics["epoch"], distill_metrics["val_acc"], "-o", color="#55A868",
               markersize=3, label="Distilled")
axes[1, 0].axhline(y=teacher_results["final_results"]["best_val_acc"], color="#4C72B0",
                   linestyle=":", alpha=0.7, label="Teacher Best Val")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Validation Accuracy")
axes[1, 0].set_title("Student Models: Validation Accuracy")
axes[1, 0].legend()

# Student comparison: training loss
axes[1, 1].plot(baseline_metrics["epoch"], baseline_metrics["train_loss"], "--o", color="#DD8452",
               markersize=3, label="Baseline (CE Loss)")
axes[1, 1].plot(distill_metrics["epoch"], distill_metrics["train_loss"], "-o", color="#55A868",
               markersize=3, label="Distilled (KD + CE Loss)")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Training Loss")
axes[1, 1].set_title("Student Models: Training Loss")
axes[1, 1].legend()

plt.suptitle("Training Dynamics Across All Models", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: reports/training_curves.png")

## 4. Distillation Loss Components

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

epochs = distill_metrics["epoch"]
distill_loss = [h["distill_loss"] for h in distill_results["epoch_history"]]
ce_loss = [h["ce_loss"] for h in distill_results["epoch_history"]]

ax.plot(epochs, distill_loss, "-o", color="#C44E52", markersize=4, label="KL Divergence Loss (soft targets)")
ax.plot(epochs, ce_loss, "-s", color="#4C72B0", markersize=4, label="Cross-Entropy Loss (hard labels)")
ax.plot(epochs, distill_metrics["train_loss"], "-^", color="#55A868", markersize=4, label="Combined Loss (α=0.7)")

ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Loss", fontsize=12)
ax.set_title("Knowledge Distillation: Loss Component Breakdown", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../reports/distillation_loss_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: reports/distillation_loss_breakdown.png")

## 5. Compression Efficiency Analysis

In [ ]:
# Compression metrics
teacher_params = teacher_results["final_results"]["num_parameters"]
student_params = distill_results["final_results"]["num_parameters"]
teacher_size = teacher_results["final_results"]["model_size_mb"]
student_size = distill_results["final_results"]["model_size_mb"]
teacher_acc = teacher_results["final_results"]["test"]["accuracy"]
distill_acc = distill_results["final_results"]["test"]["accuracy"]

compression_ratio = teacher_params / student_params
size_reduction_pct = (1 - student_size / teacher_size) * 100
accuracy_retention = distill_acc / teacher_acc * 100
f1_per_mb_teacher = teacher_results["final_results"]["test"]["f1_macro"] / teacher_size
f1_per_mb_student = distill_results["final_results"]["test"]["f1_macro"] / student_size

print("=" * 60)
print("COMPRESSION EFFICIENCY REPORT")
print("=" * 60)
print(f"Parameter Compression:  {compression_ratio:.1f}x ({teacher_params:,} → {student_params:,})")
print(f"Size Reduction:         {size_reduction_pct:.1f}% ({teacher_size:.1f} MB → {student_size:.1f} MB)")
print(f"Accuracy Retention:     {accuracy_retention:.2f}%")
print(f"F1/MB (Teacher):        {f1_per_mb_teacher:.6f}")
print(f"F1/MB (Student):        {f1_per_mb_student:.6f}")
print(f"Efficiency Gain:        {f1_per_mb_student/f1_per_mb_teacher:.1f}x better F1/MB ratio")
print("=" * 60)

In [ ]:
# Efficiency scatter plot
fig, ax = plt.subplots(figsize=(8, 6))

models_data = [
    ("Teacher (BERT-base)", teacher_size, teacher_results["final_results"]["test"]["f1_macro"], teacher_params),
    ("Student (Baseline)", baseline_results["final_results"]["model_size_mb"],
     baseline_results["final_results"]["test"]["f1_macro"], baseline_results["final_results"]["num_parameters"]),
    ("Student (Distilled)", student_size, distill_results["final_results"]["test"]["f1_macro"], student_params),
]

for name, size, f1, params in models_data:
    ax.scatter(size, f1, s=params/500000, alpha=0.7, edgecolors="black", linewidth=0.5)
    ax.annotate(name, (size, f1), textcoords="offset points", xytext=(10, 10), fontsize=9)

ax.set_xlabel("Model Size (MB)", fontsize=12)
ax.set_ylabel("Test F1 (Macro)", fontsize=12)
ax.set_title("Model Efficiency: F1 Score vs. Size\n(bubble size ∝ parameter count)", fontsize=13)
ax.set_ylim(0.975, 1.0)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../reports/efficiency_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: reports/efficiency_scatter.png")

## 6. Distillation Gain Analysis

In [ ]:
# Compare baseline vs distilled across epochs
fig, ax = plt.subplots(figsize=(10, 5))

# Compute running best for both
baseline_best = baseline_metrics["val_acc"].cummax()
distill_best = distill_metrics["val_acc"].cummax()

ax.plot(baseline_metrics["epoch"], baseline_best, "--", color="#DD8452", linewidth=2, label="Baseline (best so far)")
ax.plot(distill_metrics["epoch"], distill_best, "-", color="#55A868", linewidth=2, label="Distilled (best so far)")
ax.fill_between(distill_metrics["epoch"], baseline_best, distill_best,
                alpha=0.1, color="green", where=(distill_best >= baseline_best))
ax.fill_between(distill_metrics["epoch"], baseline_best, distill_best,
                alpha=0.1, color="red", where=(distill_best < baseline_best))

ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Best Validation Accuracy", fontsize=12)
ax.set_title("Distillation vs. Baseline: Running Best Validation Accuracy", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../reports/distillation_gain.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: reports/distillation_gain.png")

## 7. Key Takeaways

### Performance
- **Teacher (BERT-base):** 99.14% accuracy — near-ceiling performance on SNIPS
- **Student Baseline:** 98.07% — strong performance from architecture alone
- **Student Distilled:** 98.36% — distillation adds +0.29% over baseline

### Compression
- **12.3x parameter reduction** (109.5M → 8.9M)
- **91.8% size reduction** (417.7 MB → 34.1 MB)
- **99.2% accuracy retention** with distillation

### Insights
1. For well-separated 7-class intent classification, even a small transformer captures most patterns
2. Distillation provides a modest but consistent improvement (+0.29%) by transferring inter-class similarity knowledge
3. The distilled model generalizes better to test data despite lower validation accuracy — soft targets act as a regularizer
4. All training completed on CPU in reasonable time (~2.7 hours total for all experiments)